# 구매이웃 기반 N/V 조건부 ID 임베딩 변환 M2 — Dunnhumby seed 42

직전 조건부 ID 변환이 자유로운 사용자 ID 임베딩에 흡수되어 추천목록을 거의 바꾸지 못한 문제를 확인합니다.

- 학습: Dunnhumby 1~683일
- 평가: 684~690일의 신규 상품
- 보정 원천: 같은 이진 구매 그래프에서 학습되는 아이템 ID 임베딩의 1-hop 이웃 집계
- 사용자 표현: `Norm(E_u + 0.05[D_N(u) + D_V(u)])`
- N/V 변환: 각각 rank 4, 유효 사용자 전체 평균 보정 제거
- 아이템 layer-0 표현: 순수 64차원 ID 임베딩(가격·인기도 특징 없음)
- 고정: binary graph, uniform negative sampling, plain BPR, 100 epoch, 하나의 optimizer
- 비교: 동일 protocol·입력 manifest의 기존 M1@64 결과 재사용

이 실행은 역사적 개발구간 seed 42 탐색이며 통계적 유의성이나 test 일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'a7cc68889caa4090377b577a0027531326ee8095'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_neighbor_conditioned_id_transform import (
    configure_neighbor_conditioned_id_transform_run,
    preflight_summary,
    run_neighbor_conditioned_id_transform_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_neighbor_conditioned_id_transform_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_neighbor_conditioned_id_transform_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['m2']['embedding_dim'] == 64
assert summary['m2']['transform_rank'] == 4
assert summary['m2']['rho'] == 0.05
assert summary['m2']['explicit_item_features'] is False
assert summary['m2']['correction_source'] == 'same_binary_graph_one_hop_item_ID_aggregate'
assert summary['m2']['population_mean_correction_removed'] is True
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['one_training_loop_and_optimizer'] is True
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_neighbor_conditioned_id_transform_screen(cfg)

In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

print('절대지표:')
display(display_df.sort_values('model_id'))

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)